# 10 · FPGA QPSK preamble detector

**Added:** a streaming coarse detector for the known 32-symbol QPSK training sequence, in parallel with the existing raw-IQ/FIR path.

**Useful:** find a likely training-window start and score in bounded captured IQ before attempting a more complete receiver.

**Implemented:** average 16 CI8 samples at each of 32 expected symbol positions, form 31 adjacent differential products, rotate them by the known quarter-turn transitions, and compute normalized squared correlation. The FPGA accelerator exposes a memory-mapped Avalon-MM control/status interface. It is a coarse phase/CFO-tolerant detector, not a timing/carrier recovery loop, payload decoder, CRC checker, or modem receiver.

Run All is Python-only. It generates a deterministic phase-pulse example in memory and plots the host integer reference. Optional saved-capture replay, simulation, build and programming cells are off by default; no cell transmits RF.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, time
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/lessons.py").exists())
for directory in (ROOT / "scripts", ROOT / "cyberether"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))
from notebook_progress import progress
print("Python:", sys.executable)
print("Project:", ROOT)

## 1 · Build a synthetic phase-pulse training burst

This deterministic software example includes a sample offset, carrier-frequency offset, arbitrary phase and noise. It is generated in memory; no radio, FPGA or capture file is involved. The candidate detector scans are all complete 32-symbol training windows.

In [ ]:
from notebook_progress import progress
with progress("Run Python example / render graphs"):
    import numpy as np
    import matplotlib.pyplot as plt
    from rf_qpsk import synthesize
    from preamble_reference import scan

    SAMPLE_RATE = 16_000_000
    SYMBOL_US = 10
    SPS = round(SAMPLE_RATE * SYMBOL_US / 1_000_000)
    with progress("Synthesize CI8 burst and scan integer preamble score · no hardware"):
        iq = synthesize(samples=16380, symbol_us=SYMBOL_US, offset_samples=47,
                        cfo_hz=17321, phase=0.7, seed=20261021)
        result = scan(iq, sps=SPS, stride=4, threshold_q15=24576)
    scores = np.asarray(result["scores_q15"])
    starts = np.asarray(result["starts"])
    fig, axes = plt.subplots(2, 1, figsize=(12, 7), gridspec_kw={"height_ratios": [1, 2]})
    sample_ix = np.arange(min(len(iq), 1200))
    axes[0].plot(sample_ix, iq[:len(sample_ix), 0], label="I")
    axes[0].plot(sample_ix, iq[:len(sample_ix), 1], label="Q", alpha=.75)
    axes[0].set(title="Synthetic CI8 phase-pulse input", xlabel="Sample", ylabel="Signed CI8")
    axes[0].legend(); axes[0].grid(alpha=.25)
    axes[1].plot(starts, scores / 32768, color="#087f8c", lw=1)
    axes[1].axhline(result["threshold_q15"] / 32768, color="#b44", ls="--", label="Q15 threshold")
    axes[1].scatter([result["best_start"]], [result["best_score_q15"] / 32768], color="#e69f00", zorder=3, label="Best candidate")
    axes[1].set(title="Normalized differential correlation (Python integer reference)", xlabel="Candidate start sample", ylabel="Q15 score / 32768", ylim=(0, 1.04))
    axes[1].legend(); axes[1].grid(alpha=.25)
    plt.tight_layout(); plt.show()
    print({key: result[key] for key in ("schema", "processing", "sps", "window", "stride", "threshold_q15", "best_start", "best_score_q15", "detected")})

## 2 · Read the detector result

The score is the squared magnitude of the sum of 31 known-rotation differential products, normalized by 31 times their total energy and encoded as unsigned Q15 (0 to 32768). It is invariant to a common phase rotation and tolerates modest carrier offset over the training span. `min_energy` rejects weak/noise-only inputs before a high normalized score can look convincing.

The result identifies a candidate coarse start. It does **not** refine symbol timing or estimate/remove carrier offset for the payload. It does not check frame CRC, demap payload bits or replace the host modem synchronizer. The trace above is a generated example, not hardware evidence.

## 3 · Optional native design generation and export

Checkpoint 08 retains SPI capture RAM and FIR II and adds the preamble Avalon block at `0x10070000`. The native VDS image and complete Quartus project are generated by the repository helper. This only edits/generated local project files; it does not compile or program the board.

In [ ]:
GENERATE_VDS = False
if GENERATE_VDS:
    with progress("Generate checkpoint 08 native VDS project"):
        subprocess.run([sys.executable, str(ROOT / "scripts/preamble_pipeline.py"), "generate"], cwd=ROOT, check=True)
    from preamble_pipeline import STAGE
    from vds_gui_export import export_gui
    with progress("Export actual Quartus VDS GUI diagram · 14 blocks"):
        diagram = export_gui(STAGE, STAGE / "results/notebook/native-gui", 14)
    from shutil import copy2
    diagram_png = ROOT / "notebooks/diagrams/10_vds_gui.png"
    copy2(diagram, diagram_png)
    from IPython.display import Image, display
    display(Image(filename=str(diagram_png)))
    print("Native Quartus export:", diagram_png)
else:
    print("Skipped. Set GENERATE_VDS=True to generate/export the optional local design.")

### Native VDS block diagram

The diagram is exported from Quartus Visual Designer Studio, not reconstructed in Python. It shows the detector attached to the raw capture path before FIR filtering.

![Checkpoint 08 native VDS diagram](diagrams/10_vds_gui.png)

## 4 · Optional RTL simulation, compile and programming

The testbench covers the detector's register interface, start/busy/done behavior, candidate/result fields and a known training window. The build/program switches below are explicit and default off. A successful compile is not hardware qualification; programming replaces the FPGA's active image. Hardware qualification passed: nine Questa fixtures, full FPGA score-trace comparisons on generated IQ and saved RF, a fresh passive SPI capture, and the existing FIR regression. The passive capture remained below the detection threshold; this is a valid no-detection result.

In [ ]:
RUN_QUESTA = False
BUILD_SOF = False
PROGRAM_FPGA = False
if RUN_QUESTA:
    with progress("Run checkpoint 08 Questa assertions"):
        subprocess.run([sys.executable, str(ROOT / "scripts/preamble_pipeline.py"), "simulate"], cwd=ROOT, check=True)
else:
    print("Simulation skipped. Set RUN_QUESTA=True to run local Questa.")
if BUILD_SOF:
    with progress("Compile checkpoint 08 Quartus project"):
        subprocess.run([sys.executable, str(ROOT / "scripts/preamble_pipeline.py"), "build"], cwd=ROOT, check=True)
    from IPython.display import FileLink, display
    from preamble_pipeline import STAGE
    display(FileLink(str(STAGE / "iq_bridge_vds.qpf")))
    display(FileLink(str(STAGE / "results" / (STAGE.name + "-quartus-project.zip"))))
else:
    print("Build skipped. Set BUILD_SOF=True after reviewing generated sources and timing constraints.")
if PROGRAM_FPGA:
    with progress("Program checkpoint 08 · replaces active FPGA image"):
        subprocess.run([sys.executable, str(ROOT / "scripts/preamble_pipeline.py"), "program"], cwd=ROOT, check=True)
else:
    print("Programming skipped. No board access occurred.")

## 5 · Optional replay of an existing capture

This checker uploads an already saved signed CI8 file over Nios/JTAG and executes the FPGA detector. It does not transmit, capture or open a radio. Input must contain an integer number of complex samples in interleaved I,Q byte order. Run from the project root, choose a fresh output directory, and adjust the symbol period to match the saved capture.

In [ ]:
REPLAY_CAPTURE = False
CAPTURE_PATH = ROOT / "cyberether/run/example.iq"
REPLAY_SYMBOL_US = 10
if REPLAY_CAPTURE:
    output = ROOT / "cyberether/run" / ("preamble-replay-" + str(time.time_ns()))
    command = [sys.executable, str(ROOT / "scripts/check_preamble.py"),
               "--input", str(CAPTURE_PATH), "--symbol-us", str(REPLAY_SYMBOL_US),
               "--output", str(output)]
    with progress("Upload CI8 and verify FPGA preamble detector · no transmission"):
        subprocess.run(command, cwd=ROOT, check=True)
    print("Saved result:", output / "proof.json")
else:
    print("Saved-capture replay skipped. Enabled replay uses FPGA/JTAG hardware; it never starts RF capture.")

### Optional SPI/FIR-to-detector hardware page

The local preamble viewer is an explicit interactive hardware action and defaults idle. Opening it does not program a device or capture RF. Checkpoint 08 passed all three browser actions with exact FPGA/reference traces and no JavaScript errors. Generated IQ took about 50 seconds, saved RF replay 85 seconds, and a passive snapshot 21 seconds. Upload/control time dominates: the detector itself took 42 ms on 10,000 samples and 95 ms on 16,380 samples. Saved replay is historical RF data, not a new transmission. No TX jobs were started, and final transmitter readback confirmed idle with both tones disabled.

![Preamble lab live view](screenshots/10_preamble_lab.png)

In [ ]:
START_PREAMBLE_LAB = False
if START_PREAMBLE_LAB:
    with progress("Start idle preamble lab viewer on port 8772"):
        subprocess.run([sys.executable, str(ROOT / "scripts/start_preamble_lab.py"), "start"], cwd=ROOT, check=True)
    print("The viewer is idle until an explicit test or capture action.")
else:
    print("Viewer launch skipped; no browser or acquisition started.")

## Implementation limits and next work

The first detector is intentionally coarse. The raw capture reaches the detector in parallel with the FIR path so the score is calculated before filtering changes the symbol waveform. Simulation, compilation/timing, exact full-trace comparisons, saved RF replay and passive hardware capture are verified. The current image uses 64% of ALMs, 43% of M20Ks and 24/368 reported DSP units (7%). Full carrier/timing recovery, demapping, CRC/FEC, packet delivery rate and longer RF BER remain separate work.